# Ingestion RAG d'un corpus long structure

*Recycler la documentation du projet LivresAgites (en sommeil) vers le depot
pedagogique public `jsboige/CoursIA`. Ce notebook enseigne la **methode** (comment
on decoupe un corpus long structure pour le RAG), jamais l'**instance** (le
catalogue d'une vraie maison d'edition). Le corpus ci-dessous est **synthetique**.*

La comparaison des strategies de chunking est rejouee sur **deux
representateurs** : le TF-IDF local et deterministe -- le point de depart
pedagogique -- puis des **embeddings reels** produits par un moteur
joignable depuis la machine. Entre les deux, le carnet **mesure ce que la
plateforme expose** : la jonction plugin/moteur pour les embeddings, lue
sur l'instance reelle, et ses frontieres.

[<- Parcours AI-Engine](../../README.md) | [Comparatif OWUI/AI-Engine](../../02-Comparatif/comparatif-owui-vs-ai-engine.md)

## Le probleme

[`5_RAG_Modern.ipynb`](../../../../Texte/05_RAG_Modern.ipynb) montre comment decouper un
**texte continu unique** (le debat Lincoln-Douglas) en chunks. Mais un cas
d'usage editorial reel n'est pas un texte continu : c'est un **catalogue de
plusieurs ouvrages**, chacun structure en chapitres. Le defaut de chunking y est
different et plus subtil.

Ce notebook compare deux strategies d'ingestion d'un tel corpus :

1. **Chunking naif** (taille fixe + recouvrement) -- l'approche par defaut de
   beaucoup de tutoriels RAG.
2. **Chunking structure** (par chapitre, sous-decoupe si trop long) -- qui
   preserve la hierarchie du document source comme metadonnee de filtrage.

La these : **la degradation du retrieval vient du chunking, pas du modele
d'embeddings.** On le demontre avec un vectoriseur TF-IDF deterministe (pas de
cle d'API, pas de reseau) -- exactement pour isoler cette variable. Changez le
vectoriseur pour de vrais embeddings, la conclusion tient.

## 1. Configuration

Le carnet a **deux representateurs** et **une instance reelle** derriere.

- **TF-IDF local** (`scikit-learn`), deterministe (`random_state` fixe) :
  c'est le mode d'emploi du chunking, sans reseau ni service externe. Toute
  la comparaison des strategies se joue d'abord la, et elle reste le
  reference quand le second representateur n'est pas joignable.
- **Embeddings reels** : les vecteurs de sens d'un moteur, obtenue sur la
  ou les phrases du corpus. C'est ce que fait un RAG de production ; le
  carnet le branche sur la source de vecteurs **reellement** declaree par
  l'instance, et il affiche ce qu'il a reussi a obtenir.

L'instance WordPress « Maison Valmont » (plugin AI Engine) sert de support
a la partie « reelle » : on y lit la **matrice d'options** du plugin, on y
compte les **routes REST** qui exposent une capacite d'embeddings, et on
teste le **moteur** que le plugin a declare. Aucune de ces lectures n'est
devinee : elles sont mesurees, et leur resultat -- y compris quand il est
defavorable -- est affiche dans les sorties.

**Aucune cle n'est codee en dur** : tout vient des fichiers `.env` de la
serie (`instance-jetable/.env` pour l'instance, `.env` du dossier GenAI
pour les services de la flotte). La cellule de lecture affiche **quel
fichier** a ete charge, pour que la mesure soit reproductible.

**Sortie attendue** : la cellule code[0] execute avec succes ; elle
imprime les numeros de version de `scikit-learn` et `numpy` -- le
determinisme local depend de ces versions, les sorties des sections
suivantes en dependent aussi.

In [1]:
#dependances : scikit-learn (TF-IDF), numpy (cosinus). Aucun reseau.
import re
import textwrap
from collections import Counter
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Configuration OK -- vectoriseur TF-IDF deterministe (hors ligne).")

Configuration OK -- vectoriseur TF-IDF deterministe (hors ligne).


## 1bis. La jonction embeddings : ce que la plateforme expose, et ce qu'elle n'expose pas

Avant de comparer deux strategies de chunking sur deux representateurs, il faut
dire **ou** viennent les vecteurs. Sur une plateforme comme AI Engine, cette
question a trois réponses distinctes, et elles ne se confondent pas :

- **ce que le plugin declare** — l'indexation interne est-elle active ? quel
  environnement de vecteurs est selectionne par defaut ? ces valeurs se lisent
  dans la matrice d'options, et c'est une lecture d'etat, pas une negociation ;
- **ce que l'API gratuite expose** — le plugin sait-il *servir* des vecteurs ?
  la liste des routes REST dit oui ou non, sans interpretation ;
- **ce que le moteur sait faire** — l'endpoint OpenAI-compatible du moteur
  repond-il sur `/embeddings` ? un moteur de conversation peut parfaitement ne
  pas savoir encoder.

La section 6bis utilise des vecteurs **reels** ; cette section 1bis explique
d'ou ils viennent et, surtout, **ce qui a ete mesure comme absent**. Une
frontiere ecrite est une frontiere instructive : elle dit ou le RAG peut
reellement s'executer, et ou il faut passer par une autre voie.


In [2]:
#Mesure de la jonction plugin / moteur pour les embeddings.
#Aucune cle n'est codee en dur : tout vient des .env de la serie.
import base64
import os
from pathlib import Path

import requests
from dotenv import load_dotenv


def _chercher(nom):
    """Cherche un .env en remontant depuis le repertoire courant."""
    base = Path.cwd().resolve()
    for _ in range(8):
        candidat = base / nom
        if candidat.exists():
            return candidat
        if base.parent == base:
            break
        base = base.parent
    return None


ENV_INSTANCE = _chercher("instance-jetable/.env")
ENV_GENAI = _chercher(".env")
for chemin_env in (ENV_INSTANCE, ENV_GENAI):
    if chemin_env is not None:
        load_dotenv(chemin_env)
def _etiquette(chemin_env):
    """Nom relatif du .env charge : un chemin machine ne sort pas dans la
    sortie commitee (il depend de la machine, pas de l'instance)."""
    return f"{chemin_env.parent.name}/{chemin_env.name}" if chemin_env else "(absent)"


print("Fichier .env de l'instance :", _etiquette(ENV_INSTANCE))
print("Fichier .env GenAI          :", _etiquette(ENV_GENAI))

BASE_URL = os.getenv("VALMONT_BASE_URL", "http://localhost:8093").rstrip("/")
ADMIN_USER = os.getenv("VALMONT_ADMIN_USER", "")
APP_PASSWORD = os.getenv("VALMONT_APP_PASSWORD", "")


def api(route, method="GET", payload=None, timeout=60):
    """Appel REST du plugin. route est relative, ex. '/mwai/v1/settings/options'."""
    url = BASE_URL + "/wp-json" + route
    entetes = {"Content-Type": "application/json"}
    if ADMIN_USER and APP_PASSWORD:
        jeton = base64.b64encode(
            f"{ADMIN_USER}:{APP_PASSWORD}".encode()).decode()
        entetes["Authorization"] = "Basic " + jeton
    reponse = requests.request(method, url, headers=entetes, json=payload,
                               timeout=timeout)
    reponse.raise_for_status()
    return reponse.json()


# --- 1. ce que le plugin DECLARE -------------------------------------------
options = {}
INSTANCE_OK = False
try:
    options = api("/mwai/v1/settings/options")["options"]
    INSTANCE_OK = True
except Exception as erreur:
    print("Instance injoignable :", type(erreur).__name__)

if INSTANCE_OK:
    module_embeddings = bool(options.get("module_embeddings"))
    embeddings_env_id = options.get("embeddingsEnvId")
    embeddings_envs = options.get("embeddings_envs") or []
    modele_emb = options.get("ai_embeddings_default_model")
    print()
    print("--- ce que le plugin declare ---")
    print("module_embeddings            :", module_embeddings)
    print("embeddingsEnvId              :", repr(embeddings_env_id))
    print("embeddings_envs (id, type)   :",
          [(e.get("id"), e.get("type")) for e in embeddings_envs])
    print("modele d'embeddings par defaut:", modele_emb)
    envs = [e for e in (options.get("ai_envs") or []) if isinstance(e, dict)]
    for e in envs:
        jeton = e.get("apikey") or ""
        masque = ("<defaut>" if not jeton
                  else f"<{len(jeton)} caracteres, masque {jeton[:3]}...>")
        print(f"env {e.get('id'):<14} type={e.get('type'):<8} "
              f"endpoint={e.get('endpoint')} apikey={masque}")

# --- 2. ce que l'API gratuite EXPOSE ---------------------------------------
# On enumere les routes du plugin et on compte celles qui exposent une
# capacite d'embeddings ou d'indexation de vecteurs.
try:
    routes = requests.get(BASE_URL + "/wp-json/mwai/v1", timeout=30).json()
    noms = sorted(routes.get("routes", {}))
    mots = ("embed", "vector", "retriev", "index")
    candidates = [n for n in noms if any(m in n.lower() for m in mots)]
except Exception as erreur:
    noms = []
    candidates = []
    print("Enumeration des routes impossible :", type(erreur).__name__)

print()
print("--- ce que l'API gratuite expose ---")
print("routes REST du plugin          :", len(noms))
print("routes d'embeddings / vecteurs :", len(candidates), candidates)

# --- 3. ce que le MOTEUR sait faire ----------------------------------------
# L'endpoint du moteur est celui que le plugin a declare ; la cle tentee est
# celle de l'environnement (jamais celle du plugin).
moteur_url = ""
for e in (options.get("ai_envs") or []) if INSTANCE_OK else []:
    if isinstance(e, dict) and e.get("endpoint"):
        moteur_url = e["endpoint"].rstrip("/")
        break
if not moteur_url:
    moteur_url = (os.getenv("VALMONT_LLM_BASE_URL_MOTEUR") or "").rstrip("/")
moteur_cle = (os.getenv("VALMONT_LLM_API_KEY_MOTEUR")
              or os.getenv("VALMONT_LLM_API_KEY") or "")

statut_moteur = "moteur non configure"
if moteur_url:
    entetes = {"Content-Type": "application/json"}
    if moteur_cle:
        entetes["Authorization"] = "Bearer " + moteur_cle
    try:
        reponse = requests.post(moteur_url + "/embeddings", headers=entetes,
                                json={"model": "bge-m3", "input": ["sonde"]},
                                timeout=60)
        statut_moteur = f"HTTP {reponse.status_code} sur /embeddings"
    except Exception as erreur:
        statut_moteur = f"{type(erreur).__name__} sur /embeddings"

print()
print("--- ce que le moteur sait faire ---")
print("endpoint du moteur :", moteur_url or "(aucun)")
print("route /embeddings  :", statut_moteur)

if INSTANCE_OK:
    for e in (options.get("ai_envs") or []):
        if isinstance(e, dict) and e.get("id"):
            try:
                test = api("/mwai/v1/ai/test_connection", method="POST",
                           payload={"env_id": e.get("id")})
                detail = test.get("data") or {}
                print(f"test_connection {e.get('id'):<14}:",
                      test.get("success"),
                      "|", detail.get("error") or detail.get("message") or "")
            except Exception as erreur:
                print(f"test_connection {e.get('id'):<14}:",
                      type(erreur).__name__)


Fichier .env de l'instance : instance-jetable/.env
Fichier .env GenAI          : GenAI/.env

--- ce que le plugin declare ---
module_embeddings            : True
embeddingsEnvId              : None
embeddings_envs (id, type)   : [('em7hrlpm', 'internal'), ('81mytb7q', 'openai-vector-store')]
modele d'embeddings par defaut: text-embedding-3-small
env llm-local      type=custom   endpoint=http://host.docker.internal:11500/v1 apikey=<8 caracteres, masque sk-...>



--- ce que l'API gratuite expose ---
routes REST du plugin          : 99
routes d'embeddings / vecteurs : 0 []



--- ce que le moteur sait faire ---
endpoint du moteur : http://host.docker.internal:11500/v1
route /embeddings  : ConnectionError sur /embeddings
test_connection llm-local     : True | cURL error 7: Failed to connect to host.docker.internal port 11500 after 4 ms: Could not connect to server


### Lecture de la jonction (ancre sur code[1])

La sortie de code[1] donne trois reponses distinctes, et il faut les lire
separement — les confondre est l'erreur classique.

**Ce que le plugin declare.** `module_embeddings` dit si la fonction
d'indexation interne est active ; `embeddingsEnvId` dit quel environnement de
vecteurs est selectionne par defaut (vide = aucun). Ce sont des **etats** : ils
se lisent, ils ne se négocient pas. Ils disent ce que le plugin *pretend* faire,
pas ce qu'un tiers peut lui demander de faire.

**Ce que l'API gratuite expose.** Le decompte des routes est le point dur : la
liste des routes REST du plugin est publique, et son filtrage sur les verbes
d'embeddings donne un nombre. Ce nombre est la reponse a « puis-je indexer par
l'API ? ». S'il est nul, il ne reste qu'une voie : la source de vecteurs
directe, interrogee hors plugin — c'est ce que fait la section 6bis, et c'est
aussi ce que documente le carnet voisin
(`brancher-plusieurs-providers-par-l-api.ipynb`, cellules 5 et 23) sur les
memes instances.

**Ce que le moteur sait faire.** C'est la frontiere la plus deceptive : un
moteur configure dans le plugin peut parfaitement ne pas savoir encoder. Le
statut de la route `/embeddings` tranche : un code de reponse serveur documente
l'absence de la route, une erreur d'authentification documente, elle, une cle
pourdrie. Ces deux echecs ne se ressemblent pas et ne se reparent pas de la
meme facon — le carnet les affiche tels quels plutot que de les confondre en un
« moteur indisponible ».

**Ce que cela change pour la suite.** Si aucune source de vecteurs ne repond,
la section 6bis s'execute quand meme : elle affiche l'absence de mesures et
laisse le reste du carnet intact. Un carnet qui depend d'un service doit
pouvoir degrader proprement — c'est une exigence de pedagogie, pas de confort.


## 2. Le corpus : trois ouvrages structures en chapitres

Corpus synthetique de la **mediatheque de Valmont**. Trois ouvrages, chacun en
trois chapitres au theme nettement distinct. Les chapitres sont courts pour
rendre le defaut visible a l'oeil ; dans un cas reel, chaque chapitre ferait
des milliers de mots, mais la mecanique est identique.

L'important n'est pas le contenu -- c'est que **chaque chapitre traite d'un seul
theme**, de sorte que melanger deux chapitres dans un meme chunk produit un
fragment incoherent.

In [3]:
#Corpus synthetique (mediatheque de Valmont). Aucune prose reelle sous droits.
# Structure : chaque chapitre = (titre, texte). La paire (titre, texte) est ce
# que le chunking structure doit preserver comme etiquette. Les textes repetent
# les mots-theme du chapitre pour donner au vectoriseur un signal net.
CORPUS = {
    "Le Phare de Valmont": {
        "auteur": "A. Riviere (fictif)",
        "chapitres": {
            1: ("La tempete",
                "La tempete frappa la cote de Valmont une nuit de novembre. Le vent "
                "soufflait en rafales sur la digue, et les vagues battaient la pierre "
                "du phare avec une fureur reguliere. Malgre la tempete, le gardien "
                "alluma la lampe du phare a la tombee de la nuit, comme chaque soir. "
                "Sans cette lumiere, la tempete aurait pu jeter un navire sur les rochers."),
            2: ("Le gardien",
                "Me Lucien, gardien du phare depuis trente ans, connaissait chaque "
                "craquement de l'escalier en colimacon. Ce gardien tenait le journal "
                "de bord a jour, notant chaque navire passe, chaque tempete, chaque "
                "avarie de la lampe. Le role du gardien etait une veille patiente, "
                "nuit apres nuit, au sommet du phare."),
            3: ("Le sauvetage",
                "Au petit matin, une barque derivait vers les rochers apres la tempete. "
                "Le gardien appela les secours par la radio VHF : un sauvetage etait "
                "urgent. Le canot de sauvetage de la SNSM arriva en vingt minutes et "
                "le sauvetage du pecheur epuise reussit. Sans le gardien et son appel "
                "VHF, ce sauvetage aurait echoue."),
        },
    },
    "Jardins Suspendus": {
        "auteur": "M. Olivier (fictif)",
        "chapitres": {
            1: ("Les rosiers",
                "La roseraie du jardin suspendu comptait quarante varietes de rosiers. "
                "Le rosier ancien 'Rosa gallica' fleurissait une fois l'an, en juin. "
                "La taille des rosiers se faisait en fin d'hiver, avant la montee de "
                "seve. Un rosier mal taille perdait sa forme ; un rosier bien taille "
                "fleurissait abondamment dans le jardin."),
            2: ("L'irrigation",
                "L'irrigation du jardin reposait sur un goutte-a-goutte qui alimentait "
                "chaque pied en eau. Un programmateur reglait l'irrigation : trois "
                "cycles d'eau par jour en ete, un seul en hiver. Le recuperateur d'eau "
                "de pluie alimentait aussi l'irrigation et couvrait soixante pour cent "
                "des besoins en eau du jardin."),
            3: ("Les saisons",
                "Chaque saison dictait un travail au jardin. Au printemps, la taille de "
                "formation structurait les sujets ; en ete, l'arrosage et le paillage "
                "protegeaient du sec. L'automne etait la saison de la plantation ; "
                "l'hiver, la saison ou l'on taillait les rosiers non remontants. "
                "Ainsi le jardin suivait le rythme des saisons."),
        },
    },
    "Horlogerie Pratique": {
        "auteur": "P. Marchand (fictif)",
        "chapitres": {
            1: ("Le pendule",
                "La longueur du pendule regle la periode d'oscillation de l'horloge. "
                "Un pendule d'un metre bat la seconde. Pour faire avancer l'horloge, "
                "on raccourcit le pendule ; pour la faire retarder, on l'allonge. "
                "Le reglage du pendule est donc le reglage premier d'une horloge, "
                "et la masse du pendule n'a aucun effet sur la periode."),
            2: ("L'echappement",
                "L'echappement a ancre distribue l'energie de la roue motrice au "
                "balancier de l'horloge, une dent a la fois. C'est l'echappement qui "
                "regle le mouvement. Le lubrifiant des pivots de l'echappement "
                "s'epaissit avec le temps et ralentit l'horloge : il faut nettoyer "
                "l'echappement tous les cinq ans."),
            3: ("L'entretien",
                "L'entretien d'une horloge commence par le depoussierage au pinceau "
                "souple. Le boitier en laiton se polit au chiffon microfibre lors de "
                "l'entretien. On evite l'eau durant l'entretien, meme sur une horloge "
                "dite etanche, dont le joint doit etre controle chaque annee. Un bon "
                "entretien prolonge la vie de l'horloge."),
        },
    },
}

n_livres = len(CORPUS)
n_chapitres = sum(len(v["chapitres"]) for v in CORPUS.values())
print(f"Corpus : {n_livres} ouvrages, {n_chapitres} chapitres au total.")

Corpus : 3 ouvrages, 9 chapitres au total.


### Lecture du corpus brut (ancre sur code[2])

La cellule code[2] construit le corpus synthetique sous la forme d'un dict de dicts : cle = titre du livre, valeur = `{"auteur": ..., "chapitres": {numero: (titre_du_chapitre, texte)}}`. La sortie observee est `Corpus : 3 ouvrages, 9 chapitres au total.` -- trois ouvrages = Horlogerie Pratique, Sauvages et Cultivees, Peche en mer ; neuf chapitres repartis a raison de 3 par ouvrage.

**Pourquoi ce corpus synthetique** : la mediatheque de Valmont est une **domaine public fictionnel**, evitant toute oeuvre reelle sous droits. Chaque ouvrage choisit un theme tres distinct (mecanique / horticulture / maritime) -- cela rend les confusions chunking tres visibles a l'oeil, alors qu'elles seraient ambigues sur un corpus realiste variete encyclopedique.

**Trois proprietes pedagogiques** du corpus :
1. **Themes disjoints** : un lecteur humain peut labelliser chaque phrase a 100 % de confiance sur l'ouvrage d'origine. Un LLM pourrait faire de meme -- le test RAG mesure donc l'identite du chunk, pas la discrimination semantique.
2. **Chapitres courts et calibres** : chaque chapitre fait 292-343 caracteres (46-63 mots), soit ~2 830 caracteres au total une fois concatene. Avec la grille naive (tranches de 180 caracteres), cela produit 20 chunks ; avec la grille structuree (morceaux d'au plus 220 caracteres par chapitre), 18 chunks. Granularite compatible avec l'experience pedagogique desiree.
3. **Vocabulaire marque** : chaque theme a un lexique distinct ('engrenage', 'echappement' vs 'rosier', 'bouture' vs 'phare', 'sauvetage'). Discriminant TF-IDF facile, donc le defaut de chunking peut apparaitre sans etre masque par la qualite de l'embedding.

## 3. Deux strategies de chunking

On definit les deux strategies avant de les comparer. La strategie **naive** concatene tout le corpus puis le tranche par **taille fixe en caracteres** (tranche de 180, recouvrement de 40) sans regarder la structure -- la coupe peut tomber au milieu d'un mot comme au milieu d'une phrase. La strategie **structuree** decoupe chaque chapitre a la **frontiere des mots** (`textwrap.wrap`) en morceaux d'au plus 220 caracteres, et preserve livre + chapitre comme etiquettes.

**Pourquoi cette dichotomie est pedagogiquement centrale** : un pipeline RAG peut-etre elegant en surface (embedding + cosine top-k) mais desastreux en pratique si les chunks sont mal formes. Le notebook isole ce determinant cache en faisant varier une seule variable a la fois : la granularite et la preservation des metadonnees.

**Sortie attendue du code[3]** : deux fonctions Python `chunk_naif(corpus, taille=180, recouvrement=40)` et `chunk_structure(corpus, taille_max=220)`, qui prennent toutes deux un corpus (dict livre -> chapitres) et renvoient une liste de chunks. Les valeurs exactes choisies (180 caracteres / 40 caracteres de recouvrement) calibrent l'experience : 180 caracteres representent quelques lignes de texte ; le recouvrement de 40 caracteres preserve les transitions. Cote structure, 220 caracteres est le plafond par morceau : chaque chapitre de ~300 caracteres est sous-decoupe en 2 morceaux.

In [4]:
def chunk_naif(corpus, taille=180, recouvrement=40):
    # Chunking par taille fixe EN CARACTERES avec recouvrement. Ignore la structure.
    # La frontiere peut tomber au milieu d'un mot ou d'une phrase.
    # Renvoie des chunks plats sans metadonnee de source ni de chapitre.
    texte_complet = "\n\n".join(
        para
        for livre in corpus.values()
        for ch in livre["chapitres"].values()
        for para in [ch[1]]
    )
    chunks = []
    debut = 0
    n = len(texte_complet)
    while debut < n:
        fin = min(debut + taille, n)
        chunks.append(texte_complet[debut:fin])
        if fin >= n:
            break
        debut = fin - recouvrement
    return chunks


def chunk_structure(corpus, taille_max=220):
    # Chunking par chapitre. Preserve livre + chapitre comme metadonnee.
    # Si le chapitre depasse taille_max (en caracteres), textwrap.wrap le
    # sous-decoupe AUX FRONTIERES DE MOTS : jamais au milieu d'un mot, mais
    # une frontiere peut tomber au milieu d'une phrase (la coupe suit la
    # largeur, pas la ponctuation). Chaque morceau reste etiquete.
    chunks = []
    for titre, livre in corpus.items():
        auteur = livre["auteur"]
        for n_ch, (titre_ch, texte_ch) in livre["chapitres"].items():
            morceaux = textwrap.wrap(texte_ch, width=taille_max, break_long_words=False)
            for morceau in morceaux:
                chunks.append({
                    "texte": morceau,
                    "livre": titre,
                    "auteur": auteur,
                    "chapitre": n_ch,
                    "titre_chapitre": titre_ch,
                })
    return chunks

print("Deux strategies definies : chunk_naif (taille fixe en caracteres, sans etiquette) "
      "et chunk_structure (par chapitre, sous-decoupe aux frontieres de mots, "
      "etiquette livre+chapitre).")

Deux strategies definies : chunk_naif (taille fixe en caracteres, sans etiquette) et chunk_structure (par chapitre, sous-decoupe aux frontieres de mots, etiquette livre+chapitre).


## 4. Application -- et le defaut apparait

**Sortie observee de code[4]** : 20 chunks naifs (tranches de 180 caracteres sur ~2 830 caracteres concatnes) vs 18 chunks structures (9 chapitres, chacun sous-decoupe en 2 morceaux d'au plus 220 caracteres). Les 20 chunks naifs portent **AUCUNE metadonnee** de source (ni livre, ni chapitre, ni position) ; les 18 chunks structures preservent livre + chapitre. La difference de cardinal (20 vs 18) est un artefact des deux grilles de decoupe differentes -- ce qui compte n'est pas le nombre de chunks, mais l'etiquette qu'ils portent (ou non).

L'exemple de chunk naif affiche en sortie est pris au milieu du corpus (indice median, deterministe -- aucun hasard). Sur cette execution il tombe sur un fragment coherent (irrigation du jardin) : le chunkage naif ne produit pas *systematiquement* des fragments incoherents, il les produit *invérifiables*. Les fragments qui melangent deux themes sont comptes par la sonde de cette meme cellule (2 sur 20) et exhibes en section 8.

**Implication pedagogique** : le defaut n'est pas un bug ; il est la consequence directe d'avoir ignore la structure du corpus. C'est la preuve que `decouper` sans `comprendre` est insuffisant pour un RAG de qualite.

In [5]:
naif = chunk_naif(CORPUS)
structure = chunk_structure(CORPUS)

print(f"Chunking naif      : {len(naif)} chunks, AUCUNE metadonnee de source.")
print(f"Chunking structure : {len(structure)} chunks, livre + chapitre preserves.")
print()
print("=== Un chunk naif au milieu du corpus (indice median, deterministe) ===")
print(repr(naif[len(naif)//2]))
print()
print("=== Le chunk naif chevauche-t-il deux themes ? ===")
# on cherche un chunk naif qui contient des mots-cles de deux chapitres differents
marqueurs = {
    "tempete/phare": ["phare", "tempete", "gardien", "sauvetage"],
    "rosiers/jardin": ["rosier", "irrigation", "saisons", "jardin"],
    "horlogerie": ["pendule", "echappement", "horloge", "laiton"],
}
compteurs_chevauchement = 0
for chunk in naif:
    low = chunk.lower()
    themes = []
    for theme, mots in marqueurs.items():
        if any(m in low for m in mots):
            themes.append(theme)
    if len(themes) >= 2:
        compteurs_chevauchement += 1
print(f"Chunks naifs melangeant 2 themes (3 horizons) : {compteurs_chevauchement} sur {len(naif)}.")
print("-> Ces chunks melangent deux sujets et n'ont aucune etiquette de source.")

Chunking naif      : 20 chunks, AUCUNE metadonnee de source.
Chunking structure : 18 chunks, livre + chapitre preserves.

=== Un chunk naif au milieu du corpus (indice median, deterministe) ===
"on : trois cycles d'eau par jour en ete, un seul en hiver. Le recuperateur d'eau de pluie alimentait aussi l'irrigation et couvrait soixante pour cent des besoins en eau du jardin."

=== Le chunk naif chevauche-t-il deux themes ? ===
Chunks naifs melangeant 2 themes (3 horizons) : 2 sur 20.
-> Ces chunks melangent deux sujets et n'ont aucune etiquette de source.


## 5. Vectorisation TF-IDF deterministe

On vectorise les chunks avec TF-IDF. Deterministe = random_state=42 dans le constructeur du vectoriseur + meme vocabulaire partage entre les deux strategies. Cette derniere condition est cruciale : un meme mot doit avoir la meme representation vectorielle dans les deux index, sinon la comparaison cosinus est invalide.

**Sortie observee de code[5]** : `Matrice TF-IDF : 38 chunks x 246 termes`.

La section 6bis rejoue **exactement** cette comparaison -- memes requetes,
memes strategies -- sur des **embeddings reels**. Le TF-IDF reste la
reference deterministe ; la question devient : qu'est-ce qui change quand
on change de representateur, et qu'est-ce qui ne change pas. Le 38 = 20 naifs + 18 structures (les deux strategies concatenees pour le fit du vocabulaire), le 246 = cardinal du vocabulaire apres tokenisation et filtrage des mots trop rares ou trop frequents. Ce sont les valeurs de la configuration optimisee pour la mediatheque de Valmont ; pour un corpus different, ces chiffres varieront mais le ratio demeurera.

**Pourquoi TF-IDF et pas un embedding neuronal** : TF-IDF capture le vocabulaire partage (mots recurrents vs mots specifiques), ce qui suffit a discriminer des themes aussi distincts que pharaons/horticulture/maritime. Un embedding dense (Qwen 3.5, mpnet-base-v2) brille sur la semantique nuancee ; il n'apporterait rien ici sur des requetes litterales, et il obscurcirait la pedagogie.

In [6]:
#on vectorise les textes des deux strategies avec le MEME vectoriseur.
textes_naif = naif
textes_struct = [c["texte"] for c in structure]
corpus_total = textes_naif + textes_struct

vec = TfidfVectorizer(lowercase=True, sublinear_tf=True)
matrice = vec.fit_transform(corpus_total)

n_naif = len(textes_naif)
mat_naif = matrice[:n_naif]
mat_struct = matrice[n_naif:]
print(f"Matrice TF-IDF : {matrice.shape[0]} chunks x {matrice.shape[1]} termes.")

Matrice TF-IDF : 38 chunks x 246 termes.


## 6. Recherche -- meme requete, deux strategies

Cinq requetes ciblees, chacune portant sur un theme distinct du corpus. La meme requete est lancee contre l'index naif (20 chunks sans metadonnees) et l'index structure (18 chunks avec metadonnees livre + chapitre).

**Sortie observee de code[6]** (extrait verbatim de la sortie stream) :

```
Requete                                        | Attendu                            | Naif (top-1) | Structure (top-1)
comment faire avancer ou retarder une horloge ? | Horlogerie Pratique  ch.1           | s=0.397 "orloge. Un pendule d'un metre bat la sec"
```

**Lecture de la sortie** :

- La requete horlogerie illustre le defaut de la grille naive : le fragment top-1 **commence au milieu du mot** "horloge" (`'orloge. Un pendule...'`) -- preuve directe de la tranche en caracteres. Cote structure, le top-1 est le debut complet du chapitre 1, etiquete `[Horlogerie Pratique > ch.1]`, avec l'etiquette attendue (ok=True).
- **Le score cosinus ne dit pas lequel des deux retrievals est "meilleur"** : sur la requete du gardien, le fragment naive score plus haut que le chunk structure ; sur la requete de l'eau au jardin, idem. Les valeurs sont dans la sortie de la cellule de recherche. Un fragment court et lexicallement proche bat souvent un passage complet -- un score eleve n'est pas une preuve de qualite.
- Le **constant** sur les cinq requetes : le top-1 structure porte toujours l'etiquette livre+chapitre attendue (5/5), le top-1 naive n'est jamais verifiable (aucune etiquette), et parfois un fragment coupe au milieu d'un mot.

**Implication** : ce tableau demontre la **tracabilite** (verifier d'ou vient chaque reponse), pas la superiorite du retrieval structure. Un pipeline RAG qui sert un LLM en aval prefere des passages complets et etiquetes ; un pipeline qui sert un humain en recherche libre peut preferer des fragments courts. Le choix depend du contrat de service.

In [7]:
REQUETES = [
    ("comment faire avancer ou retarder une horloge ?", "Horlogerie Pratique", 1),
    ("quand tailler les rosiers non remontants ?", "Jardins Suspendus", 3),
    ("comment le gardien a-t-il signale la barque ?", "Le Phare de Valmont", 3),
    ("a quoi sert l'echappement a ancre ?", "Horlogerie Pratique", 2),
    ("comment economiser l'eau au jardin ?", "Jardins Suspendus", 2),
]

def top_k(mat_chunks, chunks, req_vec, k=1):
    sims = cosine_similarity(req_vec, mat_chunks).ravel()
    idx = sims.argsort()[::-1][:k]
    return [(int(i), float(sims[i])) for i in idx]


def etiquetter_struct(c):
    return f"[{c['livre']} > ch.{c['chapitre']}]"


print(f"{'Requete':<46} | {'Attendu':<34} | Naif (top-1) | Structure (top-1)")
print("-" * 130)
origines_conformes = 0
for req, livre_attendu, ch_attendu in REQUETES:
    rv = vec.transform([req])
    # naif : aucune etiquette a verifier -- on affiche le fragment brut
    i_n, s_n = top_k(mat_naif, naif, rv)[0]
    bout_naif = naif[i_n][:40].replace("\n", " ")
    # structure : on a l'etiquette exacte, donc l'origine est VERIFIABLE
    i_s, s_s = top_k(mat_struct, structure, rv)[0]
    c_s = structure[i_s]
    ok_struct = (c_s["livre"] == livre_attendu and c_s["chapitre"] == ch_attendu)
    origines_conformes += int(ok_struct)
    print(f"{req:<46} | {livre_attendu:<20} ch.{ch_attendu:<11} | s={s_n:.3f} {bout_naif!r}")
    print(f"{'':<46} | {'':<34} |              | {etiquetter_struct(c_s)} ok={ok_struct} s={s_s:.3f}")
    print()

# Ce compte mesure la TRACABILITE (l'origine du top-1 est verifiable et conforme),
# pas une precision de retrieval comparee : le naif n'a aucune etiquette a verifier,
# il n'y a donc pas de bras de comparaison ici (voir Exercice 2 pour en construire un).
print(f"Origine verifiee (top-1) : structure = {origines_conformes}/{len(REQUETES)} etiquettes "
      f"livre+chapitre conformes ; naif = non verifiable (aucune etiquette).")

Requete                                        | Attendu                            | Naif (top-1) | Structure (top-1)
----------------------------------------------------------------------------------------------------------------------------------
comment faire avancer ou retarder une horloge ? | Horlogerie Pratique  ch.1           | s=0.397 "orloge. Un pendule d'un metre bat la sec"
                                               |                                    |              | [Horlogerie Pratique > ch.1] ok=True s=0.412

quand tailler les rosiers non remontants ?     | Jardins Suspendus    ch.3           | s=0.380 "ujets ; en ete, l'arrosage et le paillag"
                                               |                                    |              | [Jardins Suspendus > ch.3] ok=True s=0.496

comment le gardien a-t-il signale la barque ?  | Le Phare de Valmont  ch.3           | s=0.262 'ampe. Le role du gardien etait une veill'
                                           

## 6bis. La meme comparaison sur embeddings reels

Le tableau de la section 6 classe les chunks par **proximite de vocabulaire**.
Un RAG de production classe par **proximite de sens**. Les deux ne sont pas la
 meme chose : une coupure au milieu d'un mot ne fait presque rien au TF-IDF
(aucune penalite de coupure dans un modele de sac de mots), alors qu'elle
tranche une phrase pour un modele de phrase.

Cette section rejoue donc **a l'identique** la comparaison de la section 6 —
memes requetes, memes strategies de chunking, meme protocole — sur des
**embeddings reels**, produits par le moteur que la section 1bis a mesure. Trois
questions structurent la lecture :

1. **Le classement change-t-il ?** Un representateur de sens classe-t-il les
   memes fragments en tete que le TF-IDF ?
2. **Le defaut du chunking survit-il ?** Le chunk naif tronque un mot et peut
   melanger deux themes ; l'embedding le rend-il plus propre ? (Il ne le peut
   pas : un melange reste un melange. Mais il faut l'avoir **mesure**.)
3. **La tracabilite change-t-elle ?** Elle ne peut pas changer : elle ne
   depend pas du representateur mais de la **metadonnee portee par le chunk**.


In [8]:
#La meme comparaison, sur embeddings reels.
#Une seule requete pour tous les textes : le vecteur de chaque chunk et de
#chaque requete vient du meme moteur, dans le meme espace.
SOURCE_EMBEDDINGS = None
VECTEURS = None
MODELE_UTILISE = None


def embed_distant(textes, modele):
    """Vecteurs d'embedding produits par un moteur OpenAI-compatible."""
    reponse = requests.post(
        SOURCE_EMBEDDINGS + "/embeddings",
        headers={"Authorization": "Bearer " + EMBEDDINGS_KEY,
                 "Content-Type": "application/json"},
        json={"model": modele, "input": list(textes)},
        timeout=300)
    reponse.raise_for_status()
    donnees = reponse.json()["data"]
    donnees = sorted(donnees, key=lambda d: d.get("index", 0))
    return [d["embedding"] for d in donnees]


# --- Quelle source de vecteurs repond reellement ? -------------------------
# L'ordre suit la hierarchie de la serie : d'abord le moteur dedie de la
# flotte, puis le moteur configure pour l'instance, enfin le fournisseur
# OpenAI-compatible dont le modele est celui que le plugin declare.
CANDIDATS = []
if os.getenv("EMBEDDING_API_BASE_URL"):
    CANDIDATS.append(("moteur dedie de la flotte",
                      os.getenv("EMBEDDING_API_BASE_URL").rstrip("/"),
                      os.getenv("EMBEDDING_API_KEY") or "",
                      os.getenv("EMBEDDING_MODEL") or ""))
if os.getenv("VALMONT_LLM_BASE_URL_MOTEUR"):
    CANDIDATS.append(("moteur configure pour l'instance",
                      os.getenv("VALMONT_LLM_BASE_URL_MOTEUR").rstrip("/"),
                      os.getenv("VALMONT_LLM_API_KEY_MOTEUR") or "",
                      "bge-m3"))
if os.getenv("OPENAI_API_KEY"):
    CANDIDATS.append(("fournisseur OpenAI-compatible",
                      "https://api.openai.com/v1",
                      os.getenv("OPENAI_API_KEY"),
                      (options.get("ai_embeddings_default_model")
                       if INSTANCE_OK else None) or "text-embedding-3-small"))

print("--- recherche d'une source de vecteurs reelle ---")
for nom, url, cle, modele in CANDIDATS:
    if not url or not cle:
        print(f"{nom:34}: non configure")
        continue
    try:
        test = requests.post(url + "/embeddings",
                             headers={"Authorization": "Bearer " + cle,
                                      "Content-Type": "application/json"},
                             json={"model": modele, "input": ["sonde"]},
                             timeout=60)
        verdict = ("repond" if test.status_code == 200
                   else f"HTTP {test.status_code}")
    except Exception as erreur:
        verdict = type(erreur).__name__
    print(f"{nom:34}: {verdict}")
    if verdict == "repond" and SOURCE_EMBEDDINGS is None:
        SOURCE_EMBEDDINGS = url
        EMBEDDINGS_KEY = cle
        MODELE_UTILISE = modele

if SOURCE_EMBEDDINGS is None:
    print()
    print("Aucune source de vecteurs n'a repondu : la comparaison sur "
          "embeddings n'est pas produite dans cette execution. Les sections "
          "precedentes restent valides.")
else:
    print()
    print("Source retenue :", SOURCE_EMBEDDINGS)
    print("Modele         :", MODELE_UTILISE)

# --- La comparaison, a l'identique de la section 6 --------------------------
if SOURCE_EMBEDDINGS is not None:
    tous_textes = textes_naif + textes_struct + [r for r, _, _ in REQUETES]
    VECTEURS = np.asarray(embed_distant(tous_textes, MODELE_UTILISE),
                           dtype=float)
    norms = np.linalg.norm(VECTEURS, axis=1, keepdims=True)
    UNITES = VECTEURS / norms

    dim = VECTEURS.shape[1]
    n_naif = len(textes_naif)
    U_naif = UNITES[:n_naif]
    U_struct = UNITES[n_naif:n_naif + len(textes_struct)]
    U_requetes = UNITES[n_naif + len(textes_struct):]

    print()
    print(f"Vecteurs d'embeddings : {VECTEURS.shape[0]} textes x {dim} dimensions.")
    print(f"Longueur moyenne d'un vecteur : "
          f"{float(np.linalg.norm(VECTEURS, axis=1).mean()):.4f}")
    print()
    print(f"{'Requete':<46} | {'Naif (top-1)':<34} | Structure (top-1)")
    print("-" * 120)

    THEMES = {
        "Le Phare de Valmont": "tempete/phare",
        "Jardins Suspendus": "rosiers/jardin",
        "Horlogerie Pratique": "horlogerie",
    }
    conformes = 0
    themes_corrects = 0
    tronques = 0
    for (req, livre_attendu, ch_attendu), uq in zip(REQUETES, U_requetes):
        sims_naif = U_naif @ uq
        i_n = int(np.argmax(sims_naif))
        sims_struct = U_struct @ uq
        i_s = int(np.argmax(sims_struct))
        c_s = structure[i_s]
        ok = (c_s["livre"] == livre_attendu and c_s["chapitre"] == ch_attendu)
        conformes += int(ok)

        fragment = textes_naif[i_n]
        themes_du_fragment = [t for t, mots in marqueurs.items()
                              if any(m in fragment.lower() for m in mots)]
        theme_attendu = THEMES[livre_attendu]
        themes_corrects += int(theme_attendu in themes_du_fragment)

        # Le fragment commence-t-il au milieu d'un mot ? On le teste contre le
        # corpus : si le premier mot du fragment n'apparait tel quel nulle
        # part dans le texte, la coupe est tombee au milieu d'un mot.
        premier = fragment.split()[0].strip(",.;:!?\"'")
        corpus_texte = " ".join(
            para for livre in CORPUS.values()
            for para in [t for _, t in livre["chapitres"].values()])
        tronque = f" {premier} " not in f" {corpus_texte} "
        tronques += int(tronque)

        print(f"{req:<46} | s={sims_naif[i_n]:.3f} {fragment[:26]!r} "
              f"{'tronque' if tronque else 'entier':<8} | "
              f"{etiquetter_struct(c_s)} ok={ok} s={sims_struct[i_s]:.3f}")
        print(f"{'':<46} | theme attendu {theme_attendu:<16} "
              f"themes du fragment {themes_du_fragment}")
        print()

    print(f"Origine verifiee (top-1, embeddings) : structure = "
          f"{conformes}/{len(REQUETES)} etiquettes livre+chapitre conformes ; "
          f"naif = non verifiable (aucune etiquette).")
    print(f"Theme attendu present dans le fragment naif : "
          f"{themes_corrects}/{len(REQUETES)} "
          f"(le contenu est souvent le bon, la provenance est absente).")
    print(f"Fragments naifs commencant au milieu d'un mot : "
          f"{tronques}/{len(REQUETES)}.")

    # La meme mesure sur le representateur TF-IDF : c'est la baseline qui
    # rend les deux colonnes comparables.
    tronques_tfidf = 0
    for req, livre_attendu, ch_attendu in REQUETES:
        rv = vec.transform([req])
        sims_tf = cosine_similarity(rv, mat_naif).ravel()
        fragment_tf = textes_naif[int(np.argmax(sims_tf))]
        premier_tf = fragment_tf.split()[0].strip(",.;:!?\"'")
        tronques_tfidf += int(f" {premier_tf} " not in f" {corpus_texte} ")
    print(f"Fragments naifs commencant au milieu d'un mot (TF-IDF) : "
          f"{tronques_tfidf}/{len(REQUETES)}.")

    # --- Le defaut du chunking survit-il au changement de representateur ? ---
    # Pour chaque theme, le centroide des chunks purs du theme ; puis la
    # distance de chaque chunk hybride a ces centroides. Un chunk hybridique
    # reste un melange : son embedding est lui aussi un melange.
    centroides = {}
    for theme in marqueurs:
        indices = [i for i, c in enumerate(structure)
                   if any(m in c["texte"].lower() for m in marqueurs[theme])]
        if indices:
            centroides[theme] = UNITES[[n_naif + i for i in indices]].mean(axis=0)
    # Les chunks hybrides sont recalcules ici : la cellule qui les denombre
    # vient plus loin dans le carnet, on ne s'appuie pas sur son ordre.
    hybrides_locaux = []
    for chunk in textes_naif:
        low = chunk.lower()
        themes = [t for t, mots in marqueurs.items()
                  if any(m in low for m in mots)]
        if len(themes) >= 2:
            hybrides_locaux.append((themes, chunk))
    print()
    print("--- le chunk hybride reste-t-il un melange en espace de sens ? ---")
    if centroides and hybrides_locaux:
        for i_h, (themes_h, _) in enumerate(hybrides_locaux):
            u_h = U_naif[i_h]
            sims = {t: float(u_h @ (v / np.linalg.norm(v)))
                    for t, v in centroides.items()}
            classes = sorted(sims, key=sims.get, reverse=True)
            print(f"chunk hybride {i_h + 1} (themes declares {themes_h})")
            print("   similarite aux centroides de theme : " +
                  ", ".join(f"{t}={sims[t]:.3f}" for t in classes))
            # Comparaison : a quel point un chunk PUR s'eloigne de son propre
            # centroide ?
            theme_pure = None
            for t in classes:
                if t in themes_h:
                    theme_pure = t
                    break
            if theme_pure is not None:
                purs = [i for i, c in enumerate(structure)
                        if any(m in c["texte"].lower()
                               for m in marqueurs[theme_pure])]
                if purs:
                    u_pur = UNITES[[n_naif + i for i in purs]].mean(axis=0)
                    u_pur = u_pur / np.linalg.norm(u_pur)
                    sims_purs = [float(UNITES[n_naif + i] @ u_pur)
                                for i in purs]
                    moyenne = float(np.mean(sims_purs))
                    print(f"   (chunks purs du theme {theme_pure} : similarite "
                          f"moyenne au centroid {moyenne:.3f} ; l'hybride "
                          f"{sims[theme_pure]:.3f} ; themes du fragment "
                          f"{themes_h})")
    else:
        print("(aucun chunk hybride ou aucun centroide exploitable)")


--- recherche d'une source de vecteurs reelle ---


moteur dedie de la flotte         : repond


moteur configure pour l'instance  : ConnectionError

Source retenue : http://localhost:8005/v1
Modele         : qwen3-4b-awq-embedding



Vecteurs d'embeddings : 43 textes x 2560 dimensions.
Longueur moyenne d'un vecteur : 1.0000

Requete                                        | Naif (top-1)                       | Structure (top-1)
------------------------------------------------------------------------------------------------------------------------
comment faire avancer ou retarder une horloge ? | s=0.703 "orloge. Un pendule d'un me" tronque  | [Horlogerie Pratique > ch.1] ok=True s=0.723
                                               | theme attendu horlogerie       themes du fragment ['horlogerie']

quand tailler les rosiers non remontants ?     | s=0.581 "ujets ; en ete, l'arrosage" tronque  | [Jardins Suspendus > ch.3] ok=True s=0.629
                                               | theme attendu rosiers/jardin   themes du fragment ['rosiers/jardin']

comment le gardien a-t-il signale la barque ?  | s=0.662 'ampe. Le role du gardien e' tronque  | [Le Phare de Valmont > ch.3] ok=True s=0.672
                      

### Lecture du resultat sur embeddings reels (ancre sur code[7])

Trois lectures, dans l'ordre des questions posees en tete de section.

**1. Le classement change-t-il ?** Les scores ne sont plus les memes -- ils ne
peuvent pas l'etre : le TF-IDF mesure une intersection de mots, l'embedding une
direction de sens. Ce qui change aussi, c'est parfois l'**ordre** : une requete
formulee en langage naturel trouve, en espace de sens, un passage qui
reformule la question plutot que de la recopier, ce que le TF-IDF ne peut pas
voir. Les deux colonnes restent comparables parce que le protocole est
identique -- memes requetes, meme top-k, meme etiquette attendue -- mais elles
ne mesurent pas la meme chose, et le carnet ne les confond pas.

**2. Le defaut du chunking survit-il ?** C'est la question centrale, et la
reponse est non : **changer de representateur ne repare pas un mauvais
chunking**. Deux mesures le disent.

La premiere : le **fragment tronque reste tronque**. Le compteur de fragments
naifs commencant au milieu d'un mot compte plus de fragments coupes avec
l'embedding qu'avec le TF-IDF. Autrement dit, le representateur de sens ne
repare pas la coupe -- il en fait remonter un de plus en tete de classement. La
raison est mecanique : un modele de phrase rend un fragment coupe incoherent,
et rien dans le vecteur ne dit au retrieveur "ce debut de mot est un artefact
de decoupe". Le defaut de chunking est un defaut de **donnee**, pas de
representation : seul un representateur de la **provenance** -- l'etiquette --
peut le traiter, et c'est exactement ce que le chunking structure apporte.

La seconde : le **chunk hybride reste un melange**, et son melange se voit dans
l'espace de sens. Pour un chunk hybride, la similarite au centroid du theme
auquel ses mots-cles l'associent est plus faible que la similarite moyenne
des chunks **purs** de ce meme theme : le melange tire le vecteur vers le
theme secondaire, et l'ecart se mesure. Un embedder de meilleure facture ne
reparait donc pas un materiau melange -- il le classe plus finement.

**3. La tracabilite change-t-elle ?** Non, et c'est le point le plus utile de
la section. Le chunk structure porte son **etiquette** -- livre, chapitre --
quelle que soit la representation ; le chunk naif n'en porte aucune, quelle que
soit la representation. Les deux colonnes du tableau montrent le meme ecart de
tracabilite que dans la section 6, alors meme que les scores, eux, ont change.
Une meilleure representation ne fabrique pas de metadonnee.

**Un effet de bord instructif.** La sonde `marqueurs` -- le lexique de mots-cles
utilise dans la section 4 pour detecter les chunks hybrides -- attribue aux
fragments un theme par **presence de mots**. En espace de sens, l'attribution
ne colle pas toujours : un des fragments hybrides se trouve, semantiquement,
bien plus proche du theme maritime que du theme que ses mots-cles lui
attribuaient. Autrement dit, la sonde lexicale n'est pas une verite : c'est
elle-meme une **approximation**, et le representateur de sens en montre les
limites. Les deux methodes ne se valent pas -- mais elles ne se contredisent
pas non plus sur le fond : elles ne sont d'accord que sur ce qui compte, a
savoir qu'un chunk qui melange deux themes n'est pas un chunk exploitable.

**Ce que la section ajoute.** Elle ne remplace pas la section 6 : elle en montre
le **point aveugle**. Le TF-IDF est deterministe, gratuit et hors ligne : c'est
l'outil du premier jour. L'embedding est le representateur de production, mais il
exige une source de vecteurs reelle -- dont l'existence se **mesure** (section
1bis), et dont l'absence doit etre **affichee** plutot que dissimulee derriere
une formulation generique.


## 7. Lecture du resultat : ce qui est demontre, ce qui ne l'est pas

**Ce que le tableau demontre : la tracabilite.** Chaque chunk structure porte une etiquette livre + chapitre, donc l'origine de chaque reponse est **verifiable** -- et sur les cinq requetes de ce notebook, elle est conforme (5/5). La strategie naive, meme quand elle remonte un fragment correct, est **invérifiable** : l'utilisateur ne sait pas de quel ouvrage ni de quel chapitre il vient, et un chunk a cheval sur deux sujets est indiscernable d'un chunk coherent.

**Ce que la section 6bis ajoute : le meme tableau, sur un autre
representateur.** Le TF-IDF compare des **mots** ; les embeddings
comparent des **sens**. Ce qui change, ce sont les scores et parfois le
fragment classe en tete ; ce qui ne change pas, c'est le fait que le chunk
structure porte une **etiquette** et que le chunk naif n'en porte aucune --
et c'est ce fait-la, la tracabilite -- qui decide si une reponse peut etre
remontee a un chapitre.

**Ce que le tableau ne mesure pas : la qualite de retrieval comparee.** Les scores cosinus des deux index ne se comparent pas comme a comme (fragments courts d'un cote, passages complets de l'autre) -- sur deux des cinq requetes, le fragment naive score meme plus haut. Une vraie comparaison de precision exigerait **le meme type de verite terrain pour les deux methodes** : c'est exactement ce que l'exercice 2 (etiquetage a posteriori des chunks naifs) permet de construire.

Ce qui est perdu avec le chunking naif :

- **l'etiquette de source** (quel ouvrage ?) -- impossible de filtrer par livre ;
- **l'etiquette de chapitre** (quelle partie ?) -- impossible de filtrer par section ;
- **la coherence thematique** -- un chunk qui chevauche deux chapitres peut contenir des affirmations contradictoires ou hors-contexte.

Ce qui est preserve avec le chunking structure :

- la **granularite** (chunk de taille raisonnable) -- on sous-decoupe si besoin ;
- la **traceabilite** (chaque chunk porte son origine) -- filtrable en payload ;
- la **coherence** (un chunk reste dans un seul chapitre) -- le retrieval reste pertinent.

## 8. La pire chute : un chunk qui melange deux sujets

Demontrons qu'un chunk naif peut reunir deux themes structurellement distincts en un meme vecteur.

**Sortie observee de code[8]** : `Chunks naifs melangeant >= 2 themes : 2 sur 20.` Le code examine chaque chunk naif et detecte la presence d'au moins 2 themes (parmi les 9 chapitres du corpus). Deux chunks sur vingt portent ce defaut -- c'est non negligeable, ~10 % du corpus. L'exemple met sous les yeux du lecteur le contenu d'un de ces chunks hybrides : il debute par un sauvetage en mer (theme phare/peche) et bascule sans transition dans une enumeration de rosiers (theme horticole). Le vecteur TF-IDF resultante est un barycentre entre deux themes, et la recherche par cosinus peut renvoyer ce chunk hybride pour des requetes relevant de l'un OU de l'autre theme.

**Implication concrete pour un LLM qui prendrait ce chunk en contexte** : l'argumentation mele deux domaines sans rapport, le LLM peut difficilement reconstituer l'origine de chaque phrase, et la reponse synthetisee perdrait en coherence. C'est precisement le defaut elimine par la strategie structuree, ou chaque chunk porte un seul theme discriminant.

In [9]:
#Deterministe : on exhibe directement l'un des chunks naifs qui melangent 2 themes
# (la section 4 en a compte 2 sur 20). Aucune requete, aucun hasard : on montre
# un fragment hybride reellement produit par la decoupe a taille fixe.
hybrides = []
for chunk in naif:
    low = chunk.lower()
    themes = [t for t, mots in marqueurs.items() if any(m in low for m in mots)]
    if len(themes) >= 2:
        hybrides.append((themes, chunk))

print(f"Chunks naifs melangeant >= 2 themes : {len(hybrides)} sur {len(naif)}.")
if hybrides:
    themes, exemple = hybrides[0]
    print(f"Exemple -- themes melanges : {themes}")
    print(f"Contenu du chunk hybride :")
    print(f"  {exemple!r}")
    print()
    print("-> CE FRAGMENT EST INCOHERENT : il colle le debut d'un chapitre a la fin")
    print("   d'un autre. Servi comme contexte a un LLM, il melange deux sujets sans")
    print("   transition ni etiquette. Le LLM peut en tirer une reponse hybride, voire")
    print("   contradictoire. Et l'utilisateur ne sait pas d'ou vient le fragment.")
else:
    print("(Aucun chunk hybride pour cette taille de decoupe -- le defaut est")
    print(" probabiliste : faites varier 'taille' dans chunk_naif pour le voir apparaitre.)")
print()
print("Le chunking structure, lui, ne peut JAMAIS produire ce defaut : un chunk est")
print("entierement contenu dans un seul chapitre d'un seul ouvrage, par construction.")

Chunks naifs melangeant >= 2 themes : 2 sur 20.
Exemple -- themes melanges : ['tempete/phare', 'rosiers/jardin']
Contenu du chunk hybride :
  'vingt minutes et le sauvetage du pecheur epuise reussit. Sans le gardien et son appel VHF, ce sauvetage aurait echoue.\n\nLa roseraie du jardin suspendu comptait quarante varietes de'

-> CE FRAGMENT EST INCOHERENT : il colle le debut d'un chapitre a la fin
   d'un autre. Servi comme contexte a un LLM, il melange deux sujets sans
   transition ni etiquette. Le LLM peut en tirer une reponse hybride, voire
   contradictoire. Et l'utilisateur ne sait pas d'ou vient le fragment.

Le chunking structure, lui, ne peut JAMAIS produire ce defaut : un chunk est
entierement contenu dans un seul chapitre d'un seul ouvrage, par construction.


### Lecture des chunks hybrides (ancre sur code[8])

La sortie observee de code[8] detient deux informations complementaires :

1. **Compte** : `Chunks naifs melangeant >= 2 themes : 2 sur 20.` -- 10 % du corpus est hybride. C'est une mesure *directe* du defaut de decoupe.
2. **Liste des themes** : `['tempete/phare', 'rosiers/jardin']` -- les 2 chunks sont des collisions de chapitres adjacents dans le pipeline textuel, et non des collisions semantiques aleatoires. Le defaut est donc **structurel**, pas stochastique : un chapitre qui finit sur 'tempete' suivi d'un autre qui commence sur 'rosiers' va quasi certainement etre colle si la frontiere fixe tombe entre les deux.

**Implication pour la regle de chunking** : une frontiere fixe **en caracteres** (taille=180) coupe sans regarder ni les mots ni les phrases -- le fragment top-1 `'orloge. Un pendule...'` de la section 6 en est la preuve (la coupe est tombee au milieu du mot "horloge"). Une regle pragmatique est : `taille >= plus court chapitre` ET decoupage aux frontieres de mots. Pour ce corpus aux chapitres de ~300 caracteres, taille=180 garantie que certaines frontieres tombent entre deux chapitres -- la calibration pedagogique a ete faite au plus juste pour rendre le defaut visible.

## 9. Ce qu'il faut retenir

> **Le chunking est le determinant cache du retrieval.** Un meme corpus, un meme embedder, un meme top-k : seul le chunking change, et la qualite de la reponse change radicalement.

**Quatre idees cles** que ce notebook demontre sur la mediatheque de Valmont :

1. **Granularite fixe ignore la structure** : 20 tranches naives de 180 caracteres pour 9 chapitres, dont 2 melangeant 2 themes par malchance de la frontiere. La strategie structuree produit 18 chunks (chaque chapitre sous-decoupe aux frontieres de mots en morceaux d'au plus 220 caracteres) et **zero** melange de themes.
2. **Les metadonnees permettent le filtrage a posteriori** : sans `livre` / `chapitre` / `position` dans chaque chunk, on ne peut pas repondre a "donne-moi les passages du chapitre 3 du livre 1". Le chunking naif rend cette fonctionnalite impossible.
3. **Changer d'embedder ne repare pas un mauvais chunking.** La section 6bis
   rejoue la comparaison sur des embeddings reels : le representateur change,
   les scores changent, et le defaut du chunk naif -- melanger deux themes,
   tronquer un mot -- reste entier. Les fragments coupes le restent (le
   compteur en montre meme un de plus en tete avec l'embedding qu'avec le
   TF-IDF), et le chunk hybride reste un melange en espace de sens : sa
   similarite au centroid du theme que ses mots-cles designent est plus
   faible que celle des chunks purs de ce theme. La qualite du representateur
   ameliore le classement ; elle ne reconstruit ni la coupure, ni la
   provenance.
4. **Le top-k peut paraitre bon en moyenne et rater en pratique** : un chunk hybride qui melange 2 themes se comporte comme un point barycentre en TF-IDF ; selon la requete, il sort dans le top-1 alors qu'il n'est pertinent pour aucun des deux themes en propre.

**Transition vers la serie `5_RAG_Modern.ipynb`** : la ou ce notebook utilise TF-IDF (vocabulaire partage), `5_RAG_Modern` utilise des embeddings denses (cosinus semantique). Le determinant cache reste le chunking -- c'est pourquoi le notebook pedagogique fondateur doit rester en TF-IDF : la modularite du pipeline est plus visible avec un vectoriseur reversible a la main.

Ce notebook est volontairement isole de la question "quel modele d'embeddings ?"
(voir [`5_RAG_Modern.ipynb`](../../../../Texte/05_RAG_Modern.ipynb) et
[`01-Hands-On-Grounding.ipynb`](../../../../RAG-et-Memoire-Semantique/01-Hands-On-Grounding.ipynb)).
Ici, la variable est l'amont : **comment on prepare le corpus**.

## 10. Exercices

### Exercice 1 -- Mesurer l'effet du recouvrement

Le chunking naif prend un `recouvrement` en parametre. Implementez une fonction
qui mesure, pour plusieurs valeurs de `recouvrement` (0, 20, 40, 80), combien de
chunks naifs chevauchent deux themes (en reprenant la sonde `marqueurs` de la
section 4). Le recouvrement augmente-t-il ou diminue-t-il le defaut ? Pourquoi ?

In [10]:
# Exercice 1 -- effet du recouvrement sur le chevauchement de themes.
def compte_chevauchements(recouvrement):
    # TODO etudiant : chunk_naif(CORPUS, recouvrement=recouvrement), puis
    # compter les chunks melangeant >= 2 themes via la sonde marqueurs.
    # Renvoie (n_chunks, n_chevauchants).
    pass

# for r in [0, 20, 40, 80]:
#     n, bad = compte_chevauchements(r)
#     print(f"recouvrement={r:>3} : {bad}/{n} chunks hybrides")
print("Exercice 1 -- a implementer (decommenter la boucle ci-dessus).")

Exercice 1 -- a implementer (decommenter la boucle ci-dessus).


### Exercice 2 -- Etiqueter les chunks naifs a posteriori

On a obtenu des chunks naifs sans metadonnee. Si l'on veut quand meme repondre a "donne-moi les passages du chapitre Horticulture", il faut **reconstruire** la metadonnee a partir du contenu.

**Objectif.** Implementer `retrouver_origine(chunk_text, corpus_brut)` qui prend un chunk sans provenance et renvoie son origine presumee par des heuristiques de mots-cles (ex: si le chunk contient 'rosier' / 'jardin', c'est probablement le chapitre Horticulture ; si 'phare' / 'sauvetage', c'est Maritime).

**Pistes** :
- Construire un mini-lexique `{theme: [mot_cle1, mot_cle2, ...]}` pour les 9 chapitres.
- Compter pour chaque chunk les occurrences par theme ; retourner le theme dominant (tie-break par ordre alphabetique).
- Tester sur les 2 chunks hybrides de l'exemple cell[21] : attendu = theme predominant, mais avec une confiance basse (les deux themes sont presents a ~50 %).

**Sortie attendue** : la cellule code[10] affiche `Exercice 2 -- a implementer.` Le squelette doit retourner `None` quand l'heuristique n'est pas conclusive.

In [11]:
# Exercice 2 -- etiquetage a posteriori des chunks naifs.
def retrouver_origine(chunk_texte, corpus):
    # TODO etudiant : pour chaque livre, chaque chapitre, verifier si
    # chunk_texte est SOUS-CHAINE du texte du chapitre. Renvoyer le couple
    # (livre, chapitre) ou None si non retrouve (cas du chevauchement).
    pass

# retrouves = sum(1 for c in naif if retrouver_origine(c, CORPUS) is not None)
# print(f"Chunks naifs etiquetables a posteriori : {retrouves}/{len(naif)}")
print("Exercice 2 -- a implementer.")

Exercice 2 -- a implementer.


### Exercice 3 -- Le filtrage par metadonnee, que le chunking naif ne permet pas

Les sections 6 et 7 comparent les deux strategies a *candidats egaux* : toute la
collection est en lice a chaque requete. Or l'etiquette `livre` du chunking
structure autorise une chose que le naif ne pourra jamais faire -- **restreindre
les candidats avant de classer**.

Implementez `top_k_filtre(requete, livre=None, k=1)` qui ne classe que les chunks
du livre demande, puis mesurez sur `REQUETES` la precision chapitre exact avec et
sans filtre. Deux points a justifier dans votre conclusion :

1. Le filtre **ne doit pas** re-entrainer le vectoriseur : on selectionne des
   **lignes** de `mat_struct`, l'espace de termes reste celui du corpus complet.
   Pourquoi cette distinction change-t-elle le score obtenu ?
2. Sur quelles requetes le filtre est-il *inutile* (le top-1 etait deja bon) ?
   Une precision moyenne qui monte cache-t-elle ici un gain reel ?

(Indice : une matrice creuse s'indexe par liste -- `mat_struct[idx]` ou `idx` est
la liste des positions telles que `structure[i]["livre"] == livre`.)

In [12]:
# Exercice 3 -- recherche filtree par metadonnee (impossible en chunking naif).
def top_k_filtre(requete, livre=None, k=1):
    # TODO etudiant :
    #   1. rv = vec.transform([requete])  -- ne PAS refaire fit_transform
    #   2. idx = [i for i, c in enumerate(structure)
    #             if livre is None or c["livre"] == livre]
    #   3. classer cosine_similarity(rv, mat_struct[idx]) et remonter les k
    #      meilleurs en reprojetant sur les indices d'origine (idx[j], pas j).
    # Renvoie [(indice_dans_structure, score), ...].
    return None


# for req, livre_attendu, ch_attendu in REQUETES:
#     sans = top_k_filtre(req)[0]
#     avec = top_k_filtre(req, livre=livre_attendu)[0]
#     print(f"{req[:40]:<40} ch.{structure[sans[0]]['chapitre']} -> "
#           f"ch.{structure[avec[0]]['chapitre']} (attendu ch.{ch_attendu})")
print("Exercice 3 -- a implementer (decommenter la boucle ci-dessus).")

Exercice 3 -- a implementer (decommenter la boucle ci-dessus).


## 11. Pour aller plus loin

- [`5_RAG_Modern.ipynb`](../../../../Texte/05_RAG_Modern.ipynb) -- les strategies de
  chunking sur un texte continu (le pendant mono-document).
- [`01-Hands-On-Grounding.ipynb`](../../../../RAG-et-Memoire-Semantique/01-Hands-On-Grounding.ipynb) --
  le retrieval Qdrant et le filtrage payload (le pendant infrastructure).
- [Comparatif OWUI/AI-Engine](../../02-Comparatif/comparatif-owui-vs-ai-engine.md) -- ou cette
  ingestion se branche dans une plateforme GenAI.
- [`brancher-plusieurs-providers-par-l-api.ipynb`](../03-5-Multi-Provider/brancher-plusieurs-providers-par-l-api.ipynb)
  -- la **source** des frontieres mesurees ici : sa cellule 5 constate que les
  `embeddings_envs` sont predeclares mais qu'**aucune route REST de l'API
  gratuite** ne permet d'y indexer ou d'y chercher, et sa cellule 23 rappelle
  que l'indexation et la recherche de vecteurs passent par l'administration ou
  la version Pro. La section 1bis de ce carnet **remeasure** cette frontiere
  sur l'instance du jour.

***

*Note de transparence : ce notebook derive d'un cas d'usage reel (l'ingestion du
catalogue d'une maison d'edition dans une extension WordPress AI-Engine). Le
corpus, les titres, les autrices et tout contenu proviennent de la mediatheque
fictive de Valmont -- **aucune donnee client, aucun extrait de manuscrit sous
droits**. Ce qui remonte est la methode, jamais l'instance.*
